# VC · Práctica 5 — Comparativa de detección de emociones faciales (FER-2013)

Este notebook **no entrena modelos**: carga la **CNN entrenada** (`outputs_p5/models/v1_CNN/cnn_fer_seed42.keras`) y los **modelos clásicos + resultados de DeepFace** (`outputs_p5/models/v1_otros/`) para evaluarlos sobre `test/` y generar todas las métricas, matrices de confusión y tablas comparativas.

**Índice**
1. Configuración, rutas de modelos cargados y librerías
2. Carga y exploración del conjunto de `test` (y muestra común)
3. Utilidades de evaluación y medición de latencia (1 hilo)
4. Evaluación de modelos clásicos cargados (PCA, HOG, LBP + KNN / SVM / Stacked)
5. Evaluación de la CNN cargada (`v1_CNN`)
6. Evaluación de DeepFace cargado (`v1_otros`)
7. Comparativa final (tablas, gráficos y exportación CSV)

## 1. Configuración, rutas y librerías

In [ ]:
import os, time, json, random, warnings
from pathlib import Path
import ctypes, glob, site

_nv = os.path.join(site.getsitepackages()[0], "nvidia")
for _name in ["cusolver/lib/libcusolver.so.11", "curand/lib/libcurand.so.10"]:
    for _p in glob.glob(os.path.join(_nv, _name)):
        ctypes.CDLL(_p, mode=ctypes.RTLD_GLOBAL)

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
os.environ.setdefault("MPLCONFIGDIR", "/tmp/mpl_cache")

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import tensorflow as tf
from tensorflow import keras
from tqdm.auto import tqdm

from skimage.feature import hog, local_binary_pattern
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)

warnings.filterwarnings("ignore", category=UserWarning)

# =====================  PARÁMETROS Y RUTAS DE MODELOS  =====================
DATA_DIR = Path(
    r"/home/cris/Proyectos/computer-vision/image-dataset/prac5_dataset"
)
TRAIN_DIR, TEST_DIR = DATA_DIR / "train", DATA_DIR / "test"

OUT_DIR    = Path("outputs_p5")
CNN_DIR    = OUT_DIR / "models/v1_CNN"
OTHERS_DIR = OUT_DIR / "models/v1_otros"

SEED       = 42
IMG_SIZE   = 48
CNN_PATH   = CNN_DIR / f"cnn_fer_seed{SEED}.keras"
DEEPFACE_N = 1000

random.seed(SEED); np.random.seed(SEED)
assert TEST_DIR.is_dir(), f"No encuentro test/ en {DATA_DIR}"
assert CNN_PATH.is_file(), f"No encuentro el modelo CNN entrenado en {CNN_PATH}"
assert OTHERS_DIR.is_dir(), f"No encuentro la carpeta de modelos clásicos/DeepFace en {OTHERS_DIR}"
print("CNN cargada desde:", CNN_PATH.resolve())
print("Modelos clásicos y DeepFace desde:", OTHERS_DIR.resolve())

## 2. Carga y exploración del conjunto de test

In [ ]:
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp"}

def read_gray(path: Path) -> np.ndarray:
    data = np.fromfile(str(path), dtype=np.uint8)
    img = cv2.imdecode(data, cv2.IMREAD_GRAYSCALE)
    if img is not None and img.shape != (IMG_SIZE, IMG_SIZE):
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
    return img

def load_split(split_dir: Path, class_names=None):
    if class_names is None:
        class_names = sorted(p.name for p in split_dir.iterdir() if p.is_dir())
    X, y = [], []
    for idx, cname in enumerate(class_names):
        files = sorted(f for f in (split_dir / cname).iterdir() if f.suffix.lower() in IMG_EXT)
        for f in tqdm(files, desc=f"{split_dir.name}/{cname}", leave=False):
            img = read_gray(f)
            if img is not None:
                X.append(img); y.append(idx)
    return np.stack(X).astype(np.uint8), np.array(y, dtype=np.int64), class_names

CLASS_NAMES = json.load(open(CNN_DIR / "class_names.json"))
N_CLASSES = len(CLASS_NAMES)
X_test, y_test, _ = load_split(TEST_DIR, CLASS_NAMES)

# Índice de muestra estratificada común (el mismo usado por DeepFace)
df_idx_file = OTHERS_DIR / "deepface_test_idx.npy"
if df_idx_file.is_file():
    df_idx = np.load(df_idx_file)
elif DEEPFACE_N is not None and DEEPFACE_N < len(y_test):
    sss = StratifiedShuffleSplit(n_splits=1, test_size=DEEPFACE_N, random_state=SEED)
    _, df_idx = next(sss.split(X_test, y_test))
    df_idx = np.sort(df_idx)
else:
    df_idx = np.arange(len(y_test))

X_sample, y_sample = X_test[df_idx], y_test[df_idx]
print("Clases:", CLASS_NAMES)
print(f"Test completo: {X_test.shape} | Muestra común: {X_sample.shape}")

In [ ]:
# --- Distribución de muestras en test y muestra común ---
counts = pd.DataFrame({
    "test_completo": np.bincount(y_test, minlength=N_CLASSES),
    "muestra_comun": np.bincount(y_sample, minlength=N_CLASSES),
}, index=CLASS_NAMES)
ax = counts.plot(kind="bar", figsize=(9, 4), rot=0)
for cont in ax.containers:
    ax.bar_label(cont, fontsize=8)
ax.set_title("Distribución de muestras por clase en evaluación")
ax.set_ylabel("nº de imágenes")
plt.tight_layout(); plt.show()
print(counts)

## 3. Utilidades de evaluación

In [ ]:
resultados = []

def registrar_evaluacion(modelo, familia, conjunto, y_true, y_pred, ms_img, mostrar_reporte=True, guardar_cm=True):
    acc = float(accuracy_score(y_true, y_pred))
    f1_mac = float(f1_score(y_true, y_pred, average="macro", zero_division=0))
    f1_w   = float(f1_score(y_true, y_pred, average="weighted", zero_division=0))
    resultados.append({
        "modelo": modelo,
        "familia": familia,
        "conjunto": conjunto,
        "n": int(len(y_true)),
        "accuracy": acc,
        "f1_macro": f1_mac,
        "f1_weighted": f1_w,
        "ms_img": float(ms_img),
    })
    print(f"[{conjunto}] {modelo} ({familia}) -> Acc={acc:.4f} | F1-macro={f1_mac:.4f} | F1-w={f1_w:.4f} | {ms_img:.3f} ms/img")
    if mostrar_reporte:
        print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0, digits=3))
    if guardar_cm:
        cm = confusion_matrix(y_true, y_pred, labels=range(N_CLASSES), normalize="true")
        fig, ax = plt.subplots(figsize=(5.5, 5))
        ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(
            ax=ax, cmap="Blues", colorbar=False, xticks_rotation=45, values_format=".2f"
        )
        ax.set_title(f"Matriz de confusión — {modelo} ({conjunto})")
        plt.tight_layout()
        safe_name = f"cm_{modelo.replace('+', '_').lower()}_{conjunto}.png"
        fig.savefig(OUT_DIR / safe_name, dpi=140)
        plt.show()

## 4. Evaluación de modelos clásicos cargados

In [ ]:
def extract_hog_single(img):
    return hog(
        img, orientations=9, pixels_per_cell=(8, 8),
        cells_per_block=(2, 2), block_norm="L2-Hys", feature_vector=True
    ).astype(np.float32)

def extract_lbp_single(img, radius=2, n_points=16):
    lbp = local_binary_pattern(img, n_points, radius, method="uniform")
    n_bins = n_points + 2
    hist, _ = np.histogram(lbp.ravel(), bins=n_bins, range=(0, n_bins), density=True)
    return hist.astype(np.float32)

pca = joblib.load(OTHERS_DIR / "pca_transformer.joblib")

def preparar_features_test(X_imgs):
    t0 = time.perf_counter()
    X_flat = (X_imgs.reshape(len(X_imgs), -1) / 255.0).astype(np.float32)
    X_pca  = pca.transform(X_flat)
    t_pca  = time.perf_counter() - t0

    t0 = time.perf_counter()
    X_hog = np.vstack([extract_hog_single(im) for im in X_imgs])
    t_hog = time.perf_counter() - t0

    t0 = time.perf_counter()
    X_lbp = np.vstack([extract_lbp_single(im) for im in X_imgs])
    t_lbp = time.perf_counter() - t0

    X_all = np.hstack([X_pca, X_hog, X_lbp])
    t_all = t_pca + t_hog + t_lbp
    return {
        "PCA": (X_pca, t_pca),
        "HOG": (X_hog, t_hog),
        "LBP": (X_lbp, t_lbp),
        "ALL": (X_all, t_all),
    }

feats_full   = preparar_features_test(X_test)
feats_sample = preparar_features_test(X_sample)

classic_order = ["pca_knn.joblib", "hog_knn.joblib", "lbp_knn.joblib",
                 "pca_svm.joblib", "hog_svm.joblib", "lbp_svm.joblib", "stacked.joblib"]

for fname in classic_order:
    fpath = OTHERS_DIR / fname
    if not fpath.is_file():
        continue
    bundle = joblib.load(fpath)
    clf_pipe, ftype, mname = bundle["model"], bundle["feat_type"], bundle["name"]

    # Test completo
    Xf, tf_ext = feats_full[ftype]
    t0 = time.perf_counter()
    preds_full = clf_pipe.predict(Xf)
    ms_full = ((tf_ext + (time.perf_counter() - t0)) / len(y_test)) * 1000.0
    registrar_evaluacion(mname, "Clásico", "completo", y_test, preds_full, ms_full, mostrar_reporte=(mname=="HOG+SVM"), guardar_cm=(mname=="HOG+SVM"))

    # Muestra común
    Xs, ts_ext = feats_sample[ftype]
    t0 = time.perf_counter()
    preds_samp = clf_pipe.predict(Xs)
    ms_samp = ((ts_ext + (time.perf_counter() - t0)) / len(y_sample)) * 1000.0
    registrar_evaluacion(mname, "Clásico", "muestra", y_sample, preds_samp, ms_samp, mostrar_reporte=False, guardar_cm=False)

## 5. Evaluación de la CNN cargada (`v1_CNN`)

In [ ]:
cnn = keras.models.load_model(CNN_PATH)
cnn_cfg_path = CNN_DIR / f"cnn_fer_seed{SEED}_config.json"
if cnn_cfg_path.is_file():
    print("Configuración CNN cargada:", json.load(open(cnn_cfg_path)))

def to_cnn_input(X):
    return X[..., None].astype("float32")

# Warm-up para que la compilación del grafo no distorsione la latencia
_ = cnn.predict(to_cnn_input(X_test[:16]), batch_size=16, verbose=0)

# Test completo (batch=256)
t0 = time.perf_counter()
p_test_cnn = cnn.predict(to_cnn_input(X_test), batch_size=256, verbose=0)
ms_cnn_full = ((time.perf_counter() - t0) / len(y_test)) * 1000.0
y_pred_cnn_full = p_test_cnn.argmax(1)
registrar_evaluacion("CNN", "CNN propia", "completo", y_test, y_pred_cnn_full, ms_cnn_full, mostrar_reporte=True, guardar_cm=True)

# Muestra común (1 a 1 o batch=1 para latencia realista de frame en tiempo real)
t0 = time.perf_counter()
p_samp_cnn = cnn.predict(to_cnn_input(X_sample), batch_size=1, verbose=0)
ms_cnn_samp = ((time.perf_counter() - t0) / len(y_sample)) * 1000.0
y_pred_cnn_samp = p_samp_cnn.argmax(1)
registrar_evaluacion("CNN", "CNN propia", "muestra", y_sample, y_pred_cnn_samp, ms_cnn_samp, mostrar_reporte=True, guardar_cm=True)

## 6. Evaluación de DeepFace cargado (`v1_otros`)

In [ ]:
df_preds = np.load(OTHERS_DIR / "deepface_preds.npy")
df_meta  = json.load(open(OTHERS_DIR / "deepface_meta.json"))

registrar_evaluacion(
    "DeepFace", "DeepFace", "muestra",
    y_sample, df_preds, df_meta["ms_img"],
    mostrar_reporte=True, guardar_cm=True
)

## 7. Comparativa final

In [ ]:
df_res = pd.DataFrame(resultados)
df_res.to_csv(OUT_DIR / "resultados_comparativa.csv", index=False)

print("=== TABLA 1: Evaluación en TEST COMPLETO (n = 7.178) ===")
display(df_res[df_res["conjunto"] == "completo"].sort_values("accuracy", ascending=False))

print("=== TABLA 2: Comparativa directa en MUESTRA COMÚN (n = 1.000) ===")
df_muestra = df_res[df_res["conjunto"] == "muestra"].sort_values("accuracy", ascending=False)
display(df_muestra)

# Gráfico resumen de precisión vs velocidad en la muestra común
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
m_plot = df_muestra.sort_values("accuracy", ascending=True)
y_pos = np.arange(len(m_plot))

axes[0].barh(y_pos + 0.18, m_plot["accuracy"], height=0.35, label="Accuracy")
axes[0].barh(y_pos - 0.18, m_plot["f1_macro"], height=0.35, label="F1-macro")
axes[0].set_yticks(y_pos); axes[0].set_yticklabels(m_plot["modelo"])
axes[0].set_xlim(0, 0.78); axes[0].set_title("Rendimiento (n = 1.000)")
axes[0].grid(axis="x", alpha=0.3); axes[0].legend()

axes[1].barh(y_pos, m_plot["ms_img"], height=0.55, color="tab:orange")
axes[1].set_yticks(y_pos); axes[1].set_yticklabels(m_plot["modelo"])
axes[1].set_title("Tiempo de inferencia (ms/imagen)")
axes[1].grid(axis="x", alpha=0.3)
for i, v in enumerate(m_plot["ms_img"]):
    axes[1].text(v + 0.1, i, f"{v:.2f} ms", va="center", fontsize=8)

plt.tight_layout()
fig.savefig(OUT_DIR / "comparativa_final.png", dpi=150)
plt.show()